In [ ]:
import os, sys, time
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score, f1_score
)

In [ ]:
# ── 학습 설정 ─────────────────────────────────────────────────
# 한 번에 모델에 넣을 이미지 개수
batch_size = 128

# 전체 학습 데이터 반복 횟수
epochs = 200

# 학습률 (SGD with Nesterov)
learning_rate = 0.1

# L2 정규화 강도
weight_decay = 5e-4

# Cutout 마스크 크기 (정규화 기법)
cutout_length = 16

In [ ]:
# CIFAR-10 클래스 이름
class_names = [
    "airplane",    # 0
    "automobile",  # 1
    "bird",        # 2
    "cat",         # 3
    "deer",        # 4
    "dog",         # 5
    "frog",        # 6
    "horse",       # 7
    "ship",        # 8
    "truck",       # 9
]

In [ ]:
# ── 장치 설정 (Apple Silicon MPS / CUDA / CPU) ────────────────
if torch.backends.mps.is_available():
    device = torch.device('mps')       # Apple Silicon GPU
elif torch.cuda.is_available():
    device = torch.device('cuda')      # NVIDIA GPU
else:
    device = torch.device('cpu')

print(f"사용 장치: {device}")
print(f"PyTorch 버전: {torch.__version__}")

In [ ]:
# ── Cutout 정규화 기법 ────────────────────────────────────────
# 학습 중 랜덤으로 사각형 영역을 0으로 마스킹하여 과적합 방지
class Cutout:
    def __init__(self, length=16):
        self.length = length

    def __call__(self, img):
        # img: Tensor (C, H, W)
        h, w = img.shape[1], img.shape[2]
        cy = torch.randint(0, h, (1,)).item()
        cx = torch.randint(0, w, (1,)).item()
        y1 = max(0, cy - self.length // 2)
        y2 = min(h, cy + self.length // 2)
        x1 = max(0, cx - self.length // 2)
        x2 = min(w, cx + self.length // 2)
        img = img.clone()
        img[:, y1:y2, x1:x2] = 0.0   # 마스킹
        return img


# ── CIFAR-10 정규화 값 (채널별 평균/표준편차) ─────────────────
# ImageNet과 달리 CIFAR-10 고유 통계값 사용
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)

# 학습용 변환: 랜덤 Crop + Flip + ColorJitter + Cutout (데이터 증강)
train_transform = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode='reflect'),  # 상하좌우 4px 패딩 후 32×32 랜덤 크롭
    T.RandomHorizontalFlip(),                             # 50% 확률로 좌우 뒤집기
    T.ColorJitter(brightness=0.2, contrast=0.2,
                  saturation=0.2),                        # 밝기/대비/채도 랜덤 변환
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    Cutout(cutout_length),                                # 16×16 영역 랜덤 마스킹
])
# 테스트용 변환: 정규화만 적용 (증강 없음)
test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

In [ ]:
# ── 데이터 로드 ───────────────────────────────────────────────
# CIFAR-10: 32×32 RGB 이미지, 10개 클래스
# - 학습: 50,000장
# - 테스트: 10,000장

full_train = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True,
    transform=train_transform
)
test_set = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True,
    transform=test_transform
)

# 학습 45,000 / 검증 5,000 분리 (마지막 5000개)
train_idx = list(range(45000))
val_idx   = list(range(45000, 50000))
train_set = Subset(full_train, train_idx)

# 검증 데이터는 augmentation 없이 test transform만 적용
val_raw = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=False,
    transform=test_transform
)
val_set = Subset(val_raw, val_idx)

print(f"학습 데이터: {len(train_set):,}장")
print(f"검증 데이터: {len(val_set):,}장")
print(f"테스트 데이터: {len(test_set):,}장")

train_loader = DataLoader(train_set, batch_size=batch_size,
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=256,
                          shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=256,
                          shuffle=False, num_workers=0)

In [ ]:
# ── 이미지 샘플 시각화 ────────────────────────────────────────
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
# 원본 (비정규화)
raw_ds = torchvision.datasets.CIFAR10('./data', train=True,
                                       download=False,
                                       transform=T.ToTensor())
for i, ax in enumerate(axes.flatten()):
    img, lbl = raw_ds[i]
    ax.imshow(img.permute(1,2,0).numpy())
    ax.set_title(class_names[lbl], fontsize=8)
    ax.axis('off')
plt.suptitle("CIFAR-10 샘플 이미지", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── WideResNet (WRN) 구현 ─────────────────────────────────────
# WideResNet-28-4:
# - depth=28: 총 28개 레이어
# - k=4: 채널 폭 확장 배수 (기본 채널 × 4)
# - 잔차 연결(skip connection)으로 기울기 소실 방지
# - BatchNormalization + Dropout으로 안정적인 학습
#
# 구조:
#   Conv → [Group1: 64ch] → [Group2: 128ch,stride2] → [Group3: 256ch,stride2]
#   → BN → ReLU → GlobalAvgPool → Dense(10)

class BasicBlock(nn.Module):
    """Wide Residual Block"""
    def __init__(self, in_ch, out_ch, stride=1, dropout_rate=0.3):
        super().__init__()
        self.bn1  = nn.BatchNorm2d(in_ch)
        self.c1   = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn2  = nn.BatchNorm2d(out_ch)
        self.drop = nn.Dropout(p=dropout_rate)
        self.c2   = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)

        # 입출력 채널이 다를 경우 shortcut에 1×1 conv 적용
        self.shortcut = (
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)
            if (stride != 1 or in_ch != out_ch) else None
        )

    def forward(self, x):
        out = self.c1(torch.relu(self.bn1(x)))
        out = self.c2(self.drop(torch.relu(self.bn2(out))))
        return out + (self.shortcut(x) if self.shortcut else x)


class WideResNet(nn.Module):
    """WideResNet-28-4"""
    def __init__(self, depth=28, k=4, num_classes=10, dropout_rate=0.3):
        super().__init__()
        assert (depth - 4) % 6 == 0, "depth = 6n + 4"
        n = (depth - 4) // 6   # 각 그룹의 블록 수 (28 → n=4)
        c = [16, 16*k, 32*k, 64*k]  # [16, 64, 128, 256]

        self.conv0  = nn.Conv2d(3, c[0], 3, padding=1, bias=False)
        self.group1 = self._make_group(c[0], c[1], n, stride=1, dr=dropout_rate)
        self.group2 = self._make_group(c[1], c[2], n, stride=2, dr=dropout_rate)
        self.group3 = self._make_group(c[2], c[3], n, stride=2, dr=dropout_rate)
        self.bn     = nn.BatchNorm2d(c[3])
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.fc     = nn.Linear(c[3], num_classes)

        # Kaiming He 초기화
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def _make_group(self, in_ch, out_ch, n_blocks, stride, dr):
        blocks = [BasicBlock(in_ch, out_ch, stride, dr)]
        for _ in range(n_blocks - 1):
            blocks.append(BasicBlock(out_ch, out_ch, 1, dr))
        return nn.Sequential(*blocks)

    def forward(self, x):
        x = self.conv0(x)
        x = self.group1(x)
        x = self.group2(x)
        x = self.group3(x)
        x = torch.relu(self.bn(x))
        x = self.pool(x).flatten(1)
        return self.fc(x)

In [ ]:
# ── 모델 생성 ─────────────────────────────────────────────────
model = WideResNet(depth=28, k=4, num_classes=10, dropout_rate=0.3).to(device)

param_count = sum(p.numel() for p in model.parameters())
print(f"모델: WideResNet-28-4")
print(f"총 파라미터 수: {param_count:,}")

# ── 옵티마이저: SGD with Nesterov momentum ────────────────────
# Adam보다 SGD+Nesterov가 CIFAR-10에서 일반적으로 더 좋은 성능
optimizer = optim.SGD(
    model.parameters(),
    lr=learning_rate,
    momentum=0.9,
    weight_decay=weight_decay,
    nesterov=True       # Nesterov accelerated gradient
)

# ── 학습률 스케줄: Cosine Annealing ──────────────────────────
# epochs 동안 학습률을 코사인 곡선을 따라 감소
# 끝에서 부드럽게 수렴하여 overfitting 방지
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-6
)

# ── 손실 함수: CrossEntropyLoss (다중 분류) ───────────────────
criterion = nn.CrossEntropyLoss()

In [ ]:
# ── 학습 함수 ─────────────────────────────────────────────────
def run_epoch(model, loader, criterion, device, optimizer=None):
    """한 에포크 학습 또는 평가"""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = total_correct = total_n = 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
            out  = model(x)
            loss = criterion(out, y)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss    += loss.item() * len(y)
            total_correct += (out.argmax(1) == y).sum().item()
            total_n       += len(y)

    return total_loss / total_n, total_correct / total_n


# ── 학습 실행 ────────────────────────────────────────────────
history = {'train_acc':[], 'val_acc':[], 'train_loss':[], 'val_loss':[]}
best_val_acc = 0
best_epoch   = 0
patience_cnt = 0
PATIENCE     = 35

print(f"학습 시작: WRN-28-4, {epochs}epochs, batch={batch_size}")
t0 = time.time()

for ep in range(epochs):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, device, optimizer)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion, device)
    scheduler.step()

    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch   = ep + 1
        patience_cnt = 0
        torch.save(model.state_dict(), 'best_wrn28_4.pth')  # 체크포인트 저장
        mark = "✓"
    else:
        patience_cnt += 1
        mark = " "

    ep_time = (time.time() - t0) / (ep + 1)
    eta     = ep_time * (epochs - ep - 1) / 60
    lr_now  = optimizer.param_groups[0]['lr']

    if (ep + 1) % 10 == 0 or ep < 5:
        print(f"Epoch {ep+1:3d}/{epochs} {mark} | "
              f"train_acc={train_acc:.4f} val_acc={val_acc:.4f} | "
              f"lr={lr_now:.5f} | ETA={eta:.1f}min")

    if patience_cnt >= PATIENCE:
        print(f"Early stopping at epoch {ep+1} (best: {best_epoch})")
        break

print(f"\n총 학습 시간: {(time.time()-t0)/60:.1f}분")
print(f"최고 val_acc: {best_val_acc:.4f} (epoch {best_epoch})")

In [ ]:
# ── 학습 곡선 시각화 ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_acc'], label='Train Accuracy', color='blue')
axes[0].plot(history['val_acc'],   label='Val Accuracy',   color='orange')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_loss'], label='Train Loss', color='blue')
axes[1].plot(history['val_loss'],   label='Val Loss',   color='orange')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('WideResNet-28-4 Training History', fontsize=13)
plt.tight_layout()
plt.savefig('cifar10_training_curve.png', dpi=150)
plt.show()

In [ ]:
# ── 최종 평가 (Best Checkpoint 로드) ──────────────────────────
model.load_state_dict(torch.load('best_wrn28_4.pth', map_location=device))
model.eval()

all_preds = []
all_true  = []

with torch.no_grad():
    for x, y in test_loader:
        preds = model(x.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y.numpy())

y_pred = np.array(all_preds)
y_test = np.array(all_true)

accuracy    = accuracy_score(y_test, y_pred)
macro_f1    = f1_score(y_test, y_pred, average='macro')
weighted_f1 = f1_score(y_test, y_pred, average='weighted')

print("\n===== Final Evaluation Metrics =====")
print(f"Accuracy           : {accuracy:.4f}")
print(f"Macro F1-score     : {macro_f1:.4f}")
print(f"Weighted F1-score  : {weighted_f1:.4f}")

In [ ]:
# ── Classification Report ─────────────────────────────────────
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

# ── Confusion Matrix ──────────────────────────────────────────
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title(f"CIFAR-10 Confusion Matrix — WideResNet-28-4\nMacro F1: {macro_f1:.4f}",
          fontsize=13)
plt.tight_layout()
plt.savefig('cifar10_confusion_matrix.png', dpi=150)
plt.show()